# Apply trained model on test events

This code reads the .joblib file with the trained Machine Learning model and deploys it to previously made feature tables for test events, gammas, protons and SMONLY protons. Creates one .txt file for each event and stores it in the "output" directory.
A cut on distances and pe number can be selected as well. In the txt all probabilities of the station are saved with their real label (1 for muons, 0 for non muons).

- Specify feature tables directory of the test events
- Specify save path for txt files
- Set minimum and maximum distance as well as minimum pe number
- Specify the function that deploys the model (should be one for each model)
- **MODEL MUST BE THE SAME FOR ALL TEST EVENTS**

# Imports

In [1]:
import sys
sys.path.append("../")
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import glob
import os
from py_code import config_loader 
from py_code import output_files_maker as ofp
from py_code import load_model

# Load Config File

In [2]:
conf = config_loader.load_config("config.yaml")
DAT_path_protons = conf["DAT_path_protons"]
DAT_path_gammas = conf["DAT_path_gammas"]

TRAIN_DAT_MIN = conf["TRAIN_DAT_MIN"]
TRAIN_DAT_MAX = conf["TRAIN_DAT_MAX"]

TEST_PROTONS_DAT_MIN = conf["TEST_PROTONS_DAT_MIN"]
TEST_PROTONS_DAT_MAX = conf["TEST_PROTONS_DAT_MAX"]

TEST_GAMMAS_DAT_MIN = conf["TEST_GAMMAS_DAT_MIN"]
TEST_GAMMAS_DAT_MAX = conf["TEST_GAMMAS_DAT_MAX"]

traces_dir = conf["traces_dir"]
feats_dir =  conf["feats_dir"]
output_dir = conf["output_dir"]
models_dir = conf["models_dir"]
results_dir = conf["results_dir"]
pictures_dir = conf["pictures_dir"]

params_dir = conf["params_dir"]
cut_dir = conf["cut_dir"]
ADC_dir = conf["ADC_dir"]
array_dir = conf["array_dir"]

features_cut_dir = conf["features_cut_dir"]
training_selection_dir = conf["training_selection_dir"]
testing_selection_dir = conf["testing_selection_dir"]

tank_type = conf["tank_type"]
FF = conf["FF"]
time_window = conf["time_window"]
min_energy_reco = conf["min_energy_reco"]
max_energy_reco = conf["max_energy_reco"]
min_theta = conf["min_theta"]
max_theta = conf["max_theta"]
intended_cut_on_pes = conf["intended_cut_on_pes"]
mpmt_or_8inches = conf["mpmt_or_8inches"]
ADC_MHz = conf["ADC_MHz"]
ML_model = conf["ML_model"]

train_min_st_dist = conf["train_min_st_dist"]
train_max_st_dist = conf["train_max_st_dist"]
train_pe_min = conf["train_pe_min"]

test_min_st_dist = conf["test_min_st_dist"]
test_max_st_dist = conf["test_max_st_dist"]
test_pe_min = conf["test_pe_min"]

FF_mask = conf["FF_mask"]
tank_type = conf["tank_type"]

In [3]:
PROJECT_DIR = Path.cwd().parent

# Create directories and parent directories

In [4]:
train_label = f"{params_dir}_{features_cut_dir}_{training_selection_dir}"
test_label = f"{params_dir}_{features_cut_dir}_{training_selection_dir}_{testing_selection_dir}"
model_save_dir = f"{PROJECT_DIR}/{models_dir}/{array_dir}/{params_dir}/{features_cut_dir}/{training_selection_dir}/"
feat_imp_save_dir = f"{PROJECT_DIR}/{models_dir}/{array_dir}/{params_dir}/{features_cut_dir}/{training_selection_dir}/"

# Specify read and save directories for test gammas
GAMMAS_feats_test_dir = Path(f"{PROJECT_DIR}/{feats_dir}/{array_dir}/test_gammas/{params_dir}/{features_cut_dir}/")
GAMMAS_probs_save_dir = Path(f"{PROJECT_DIR}/{output_dir}/{array_dir}/test_gammas/{params_dir}/{features_cut_dir}/{testing_selection_dir}/")

# Specify read and save directories for test protons
PROTONS_feats_test_dir = Path(f"{PROJECT_DIR}/{feats_dir}/{array_dir}/test_protons/{params_dir}/{features_cut_dir}/")
PROTONS_probs_save_dir = Path(f"{PROJECT_DIR}/{output_dir}/{array_dir}/test_protons/{params_dir}/{features_cut_dir}/{testing_selection_dir}/")

# Specify read and save directories for test protons SM-ONLY
SMONLY_feats_test_dir = Path(f"{PROJECT_DIR}/{feats_dir}/{array_dir}/test_protons_SMONLY/{params_dir}/{features_cut_dir}/")
SMONLY_probs_save_dir = Path(f"{PROJECT_DIR}/{output_dir}/{array_dir}/test_protons_SMONLY/{params_dir}/{features_cut_dir}/{testing_selection_dir}/")

GAMMAS_probs_save_dir.mkdir(parents=True, exist_ok=True)
PROTONS_probs_save_dir.mkdir(parents=True, exist_ok=True)
SMONLY_probs_save_dir.mkdir(parents=True, exist_ok=True)


for f in glob.glob(str(GAMMAS_probs_save_dir) + "*/.txt"):
    os.remove(f)

for f in glob.glob(str(PROTONS_probs_save_dir) + "/*.txt"):
    os.remove(f)
    
for f in glob.glob(str(SMONLY_probs_save_dir) + "/*.txt"):
    os.remove(f)

test_module = load_model.test_model(model_name = ML_model)

# Apply on Gammas

In [5]:
for parquet_file in GAMMAS_feats_test_dir.glob("*.parquet"):
    name = parquet_file.stem   
    print(f"Processing file: {name}") 
    parquet_file_name = str(parquet_file)
    feats_path = parquet_file_name

    y_te, p_te, info_test = test_module.test(
                                             path = parquet_file_name,
                                             model_save_path = model_save_dir + "model_" + train_label + ".joblib",
                                             min_st_dist = test_min_st_dist,
                                             max_st_dist = test_max_st_dist,
                                             pe_min = test_pe_min
                                             )

    y_te_masked, p_te_masked = ofp.build_output_file(
                                                    txt_name = name + ".txt",
                                                    output_results_path = str(GAMMAS_probs_save_dir),
                                                    y_te = y_te,
                                                    p_te = p_te,
                                                    info = info_test,
                                                    tank_type = tank_type,
                                                    FF_mask = FF_mask
                                                    )

Processing file: DAT100309_2
Processing file: DAT100206_1
Processing file: DAT100372_3
Processing file: DAT100467_2
Processing file: DAT100078_8
Processing file: DAT100075_1
Processing file: DAT100101_3
Processing file: DAT100343_4
Processing file: DAT100163_13
Processing file: DAT100338_5
Processing file: DAT100499_11
Processing file: DAT100269_6
Processing file: DAT100255_8
Processing file: DAT100212_7
Processing file: DAT100081_2
Processing file: DAT100124_2
Processing file: DAT100473_4
Processing file: DAT100442_3
Processing file: DAT100115_5
Processing file: DAT100493_1
Processing file: DAT100061_7
Processing file: DAT100289_3
Processing file: DAT100357_2
Processing file: DAT100471_10
Processing file: DAT100258_1
Processing file: DAT100163_9
Processing file: DAT100050_4
Processing file: DAT100212_3
Processing file: DAT100269_2
Processing file: DAT100366_1
Processing file: DAT100289_7
Processing file: DAT100386_4
Processing file: DAT100258_5
Processing file: DAT100439_6
Processing 

# Apply on Protons

In [6]:
for parquet_file in PROTONS_feats_test_dir.glob("*.parquet"):
    name = parquet_file.stem   
    print(f"Processing file: {name}") 
    parquet_file_name = str(parquet_file)
    feats_path = parquet_file_name

    y_te, p_te, info_test = test_module.test(
                                             path = parquet_file_name,
                                             model_save_path = model_save_dir + "model_" + train_label + ".joblib",
                                             min_st_dist = test_min_st_dist,
                                             max_st_dist = test_max_st_dist,
                                             pe_min = test_pe_min
                                             )

    y_te_masked, p_te_masked = ofp.build_output_file(
                                                    txt_name = name + ".txt",
                                                    output_results_path = str(PROTONS_probs_save_dir),
                                                    y_te = y_te,
                                                    p_te = p_te,
                                                    info = info_test,
                                                    tank_type = tank_type,
                                                    FF_mask = FF_mask
                                                    )

Processing file: DAT1400342_2
Processing file: DAT1405045_19
Processing file: DAT1400353_19
Processing file: DAT1405510_17
Processing file: DAT1405464_21
Processing file: DAT1405024_20
Processing file: DAT1405014_10
Processing file: DAT1400338_18
Processing file: DAT1400359_21
Processing file: DAT1405099_15
Processing file: DAT1400369_11
Processing file: DAT1405071_14
Processing file: DAT1405041_24
Processing file: DAT1400381_10
Processing file: DAT1405097_10
Processing file: DAT1405042_8
Processing file: DAT1405039_9
Processing file: DAT1405005_7
Processing file: DAT1400394_34
Processing file: DAT1400373_5
Processing file: DAT1400437_13
Processing file: DAT1405506_13
Processing file: DAT1400314_14
Processing file: DAT1400399_11
Processing file: DAT1405020_6
Processing file: DAT1405069_15
Processing file: DAT1400397_14
Processing file: DAT1400409_3
Processing file: DAT1405011_1
Processing file: DAT1400341_20
Processing file: DAT1405148_17
Processing file: DAT1400367_3
Processing file: 

# Apply on Protons SMONLY

In [7]:
for parquet_file in SMONLY_feats_test_dir.glob("*.parquet"):
    name = parquet_file.stem   
    print(f"Processing file: {name}") 
    parquet_file_name = str(parquet_file)
    feats_path = parquet_file_name

    y_te, p_te, info_test = test_module.test(
                                             path = parquet_file_name,
                                             model_save_path = model_save_dir + "model_" + train_label + ".joblib",
                                             min_st_dist = test_min_st_dist,
                                             max_st_dist = test_max_st_dist,
                                             pe_min = test_pe_min
                                             )
    
    y_te_masked, p_te_masked = ofp.build_output_file(
                                                    txt_name = name + ".txt",
                                                    output_results_path = str(SMONLY_probs_save_dir),
                                                    y_te = y_te,
                                                    p_te = p_te,
                                                    info = info_test,
                                                    tank_type = tank_type,
                                                    FF_mask = FF_mask
                                                    )

Processing file: DAT1400393_6
Processing file: DAT1400342_4
Processing file: DAT1400339_5
Processing file: DAT1400369_23
Processing file: DAT1405008_8
Processing file: DAT1400439_24
Processing file: DAT1400409_14
Processing file: DAT1405034_6
Processing file: DAT1400332_12
Processing file: DAT1405464_13
Processing file: DAT1405024_12
Processing file: DAT1405005_1
Processing file: DAT1400437_21
Processing file: DAT1405135_20
Processing file: DAT1405071_26
Processing file: DAT1405041_16
Processing file: DAT1400357_16
Processing file: DAT1400373_3
Processing file: DAT1405059_17
Processing file: DAT1405219_6
Processing file: DAT1400425_18
Processing file: DAT1400356_2
Processing file: DAT1405506_21
Processing file: DAT1400409_5
Processing file: DAT1405546_1
Processing file: DAT1405056_8
Processing file: DAT1400367_5
Processing file: DAT1400397_26
Processing file: DAT1405057_12
Processing file: DAT1400324_12
Processing file: DAT1405146_24
Processing file: DAT1400404_8
Processing file: DAT14